In [2]:
from sklearn import svm
x = [[0,0],[1,1]]
y= [0,1] 
clf = svm.SVC(C=0.1)
clf.fit(x,y)
pred = clf.predictt([[2., 2., 1.]]) 
print(pred)



AttributeError: 'SVC' object has no attribute 'predictt'

In [3]:
import pandas as pd
import numpy as np

# Vector w y sesgo b dados en el enunciado
w = np.array([1/3, 1, -3, -2, 5/4], dtype=float)
b = 2/3

def clasificar_muestra(x):
    """
    Recibe una muestra x de 5 características y retorna:
    - el valor de f(x)
    - la clase asignada
    """
    fx = np.dot(w, x) + b
    clase = 1 if fx >= 0 else 0
    return fx, clase

def clasificar_csv(archivo_entrada, archivo_salida="resultado_lineal.csv"):
    """
    Lee un archivo CSV con 5 columnas numéricas (una muestra por fila),
    calcula f(x) y asigna la clase.
    """
    df = pd.read_csv(archivo_entrada)

    # Verificar que tenga exactamente 5 columnas
    if df.shape[1] != 5:
        raise ValueError("El archivo CSV debe tener exactamente 5 columnas.")

    X = df.to_numpy(dtype=float)

    valores_fx = []
    clases = []

    for x in X:
        fx, clase = clasificar_muestra(x)
        valores_fx.append(fx)
        clases.append(clase)

    df["f(x)"] = valores_fx
    df["clase"] = clases

    df.to_csv(archivo_salida, index=False)
    print(f"Clasificación completada. Archivo guardado como: {archivo_salida}")
    print(df)

if __name__ == "__main__":
    # Cambia el nombre por el archivo que vayas a usar
    clasificar_csv("muestras_lineales.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'muestras_lineales.csv'

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import multivariate_normal

# Parámetros de las clases
mu1 = np.array([0, 0], dtype=float)
sigma1 = np.array([[1, 0.5],
                   [0.5, 1]], dtype=float)

mu2 = np.array([4, 4], dtype=float)
sigma2 = np.array([[1, -0.3],
                   [-0.3, 1]], dtype=float)

mu3 = np.array([-4, 3], dtype=float)
sigma3 = np.array([[0.5, 0],
                   [0, 2]], dtype=float)

# Distribuciones gaussianas
gauss1 = multivariate_normal(mean=mu1, cov=sigma1)
gauss2 = multivariate_normal(mean=mu2, cov=sigma2)
gauss3 = multivariate_normal(mean=mu3, cov=sigma3)

def clasificar_muestra(x):
    """
    Calcula la probabilidad de la muestra x en cada clase
    y asigna la clase con mayor densidad.
    """
    p1 = gauss1.pdf(x)
    p2 = gauss2.pdf(x)
    p3 = gauss3.pdf(x)

    probabilidades = [p1, p2, p3]
    clase = np.argmax(probabilidades) + 1  # +1 para que las clases sean 1,2,3

    return p1, p2, p3, clase

def clasificar_csv(archivo_entrada, archivo_salida="resultado_gaussiano.csv"):
    """
    Lee un CSV con 2 columnas numéricas (x,y),
    calcula la densidad en cada clase y asigna la más probable.
    """
    df = pd.read_csv(archivo_entrada)

    if df.shape[1] != 2:
        raise ValueError("El archivo CSV debe tener exactamente 2 columnas.")

    X = df.to_numpy(dtype=float)

    p1_list = []
    p2_list = []
    p3_list = []
    clases = []

    for x in X:
        p1, p2, p3, clase = clasificar_muestra(x)
        p1_list.append(p1)
        p2_list.append(p2)
        p3_list.append(p3)
        clases.append(clase)

    df["P(Clase1)"] = p1_list
    df["P(Clase2)"] = p2_list
    df["P(Clase3)"] = p3_list
    df["clase_asignada"] = clases

    df.to_csv(archivo_salida, index=False)
    print(f"Clasificación completada. Archivo guardado como: {archivo_salida}")
    print(df)

if __name__ == "__main__":
    clasificar_csv("muestras_gaussianas.csv")

In [ ]:
import pandas as pd
import numpy as np

def distancia_euclidea(a, b):
    """
    Calcula la distancia euclídea entre dos vectores.
    """
    return np.sqrt(np.sum((a - b) ** 2))

def encontrar_mas_cercanas(archivo_consulta, archivo_referencia, archivo_salida="resultado_distancias.csv"):
    """
    Para cada muestra del archivo_consulta, busca la muestra más cercana
    dentro del archivo_referencia.
    """
    df_consulta = pd.read_csv(archivo_consulta)
    df_referencia = pd.read_csv(archivo_referencia)

    X_consulta = df_consulta.to_numpy(dtype=float)
    X_referencia = df_referencia.to_numpy(dtype=float)

    # Verificar que ambas tengan el mismo número de columnas
    if X_consulta.shape[1] != X_referencia.shape[1]:
        raise ValueError("Ambos archivos deben tener el mismo número de columnas.")

    resultados = []

    for i, muestra in enumerate(X_consulta):
        mejor_indice = -1
        mejor_distancia = float("inf")

        for j, ref in enumerate(X_referencia):
            dist = distancia_euclidea(muestra, ref)
            if dist < mejor_distancia:
                mejor_distancia = dist
                mejor_indice = j

        resultados.append({
            "indice_muestra_consulta": i,
            "indice_muestra_mas_cercana": mejor_indice,
            "distancia_minima": mejor_distancia
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados.to_csv(archivo_salida, index=False)
    print(f"Proceso completado. Archivo guardado como: {archivo_salida}")
    print(df_resultados)

if __name__ == "__main__":
    encontrar_mas_cercanas("muestras_consulta.csv", "muestras_referencia.csv")